# 🎬 Wan 2.2 I2V — ComfyUI on Colab (A100) · **Hugging Face weights, Drive LoRAs**

Same workflow as `wan22_i2v_comfyui_colab.ipynb`, with the storage split fixed:

| What | Size | Where it lives | Why |
|---|---|---|---|
| Diffusion models, text encoder, VAE, CLIP vision | ~34 GB | **Hugging Face** → runtime local disk, re-pulled each session | Too big to keep in Drive; `hf_transfer` re-fetches all of it in ~3–8 min |
| **LoRAs** | ~0.6–1.2 GB | **Google Drive**, read in place | Small, and often your own — worth keeping durably |
| Renders & input images | small | **Google Drive** | You want to keep these |

Net Drive usage: **your LoRAs and renders only.** The 34 GB of base weights never touch it.

Wan 2.2 14B uses a dual-expert (MoE) design: a **high-noise** model handles early denoising steps and a **low-noise** model finishes — better motion coherence than Wan 2.1, at the cost of loading two diffusion models.

| Component | File | ComfyUI folder | Source |
|---|---|---|---|
| High-noise expert | `wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors` (~13 GB) | `diffusion_models` | HF |
| Low-noise expert | `wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors` (~13 GB) | `diffusion_models` | HF |
| Text encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` (~6 GB) | `text_encoders` | HF |
| VAE | `wan_2.1_vae.safetensors` (~242 MB) | `vae` | HF |
| CLIP vision* | `clip_vision_h.safetensors` (~1.2 GB) | `clip_vision` | HF |
| Lightning LoRAs | `*_I2V_*_high/low_noise_*.safetensors` | `loras` | **Drive** |

> ⚠️ Use **`wan_2.1_vae.safetensors`** — *not* the 2.2 VAE — for the 14B I2V workflow.
> *CLIP vision is only needed for Wan 2.1-style I2V graphs; the native Wan 2.2 14B I2V template does not require it, but it's kept for compatibility.

**Setup:** `Runtime → Change runtime type → A100 GPU`. Colab A100 = 40 GB VRAM; fp8 I2V uses ~30 GB.

## Step 1 — Verify GPU

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
if 'A100' in gpu:
    print('✅ A100 confirmed — up to 161 frames (~10s) is safe')
elif 'L4' in gpu:
    print('⚠️  L4 — keep frames ≤ 113 (~7s) and use 480p')
elif 'T4' in gpu:
    print('⚠️  T4 (16 GB) is too small for 14B fp8. Switch to A100: Runtime → Change runtime type → A100 GPU')
else:
    print('⚠️  Recommended: A100. Runtime → Change runtime type → A100 GPU')

## Step 2 — Mount Drive & set paths

Drive holds `loras/`, `output/` and `input_images/`. The ~34 GB of base weights go to
`/content/models` on the runtime's local disk, which is wiped when the runtime recycles —
that's the point: they are re-pulled from Hugging Face instead of occupying Drive.

In [ ]:
import os, shutil
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ComfyUI_Wan'

# Big weights: local disk, re-fetched from Hugging Face each session.
HF_MODELS_DIR = '/content/models'
# LoRAs: durable, in Drive, read in place (small enough that FUSE reads are fine).
LORA_DIR   = f'{DRIVE_BASE}/models/loras'
OUTPUT_DIR = f'{DRIVE_BASE}/output'
INPUT_DIR  = f'{DRIVE_BASE}/input_images'

HF_SUBDIRS = ('diffusion_models', 'text_encoders', 'vae', 'clip_vision')

for d in [f'{HF_MODELS_DIR}/{s}' for s in HF_SUBDIRS] + [LORA_DIR, OUTPUT_DIR, INPUT_DIR]:
    os.makedirs(d, exist_ok=True)

free = shutil.disk_usage('/content').free / 1024**3
print('✅ Paths ready')
print(f'   base weights (HF → local) → {HF_MODELS_DIR}')
print(f'   loras        (Drive)      → {LORA_DIR}')
print(f'   output       (Drive)      → {OUTPUT_DIR}')
print(f'   input        (Drive)      → {INPUT_DIR}')
print(f'\n   local disk free: {free:.0f} GB')
if free < 45:
    print('⚠️  The base weights need ~34 GB plus download headroom.')
    print('   Runtime → Disconnect and delete runtime gives you a clean disk.')

## Step 3 — Install ComfyUI + custom nodes

Native Wan 2.2 support is built into recent ComfyUI core. Adds ComfyUI-Manager and VideoHelperSuite (for video preview/save nodes).

In [ ]:
import os, subprocess
os.chdir('/content')

# ComfyUI — clone fresh if missing/corrupt, else update
if os.path.exists('/content/ComfyUI'):
    ok = subprocess.run(['git', 'rev-parse', '--git-dir'], cwd='/content/ComfyUI',
                        capture_output=True).returncode == 0
    if ok:
        !cd /content/ComfyUI && git pull -q
        print('✅ ComfyUI updated')
    else:
        !rm -rf /content/ComfyUI && git clone -q https://github.com/comfyanonymous/ComfyUI.git
        print('✅ ComfyUI re-cloned (was corrupt)')
else:
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
    print('✅ ComfyUI cloned')

# Core requirements (flag avoids reinstalling every session)
if not os.path.exists('/content/comfyui_reqs_installed'):
    !pip install -q -r /content/ComfyUI/requirements.txt
    open('/content/comfyui_reqs_installed', 'w').close()
    print('✅ Requirements installed')
else:
    print('✅ Requirements already installed')

# Custom nodes
for name, repo in [('ComfyUI-Manager', 'https://github.com/ltdrdata/ComfyUI-Manager.git'),
                   ('ComfyUI-VideoHelperSuite', 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')]:
    path = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(path):
        !git clone -q {repo} {path}
        if os.path.exists(f'{path}/requirements.txt'):
            !pip install -q -r {path}/requirements.txt
        print(f'✅ {name} installed')
    else:
        !cd {path} && git pull -q
        print(f'✅ {name} ready')

# ffmpeg (for video utilities)
if subprocess.run(['which', 'ffmpeg'], capture_output=True).returncode != 0:
    !apt-get install -y -q ffmpeg
print('\n✅ All installs complete')

## Step 4 — Link both locations into ComfyUI

ComfyUI only ever sees `models/<folder>`, so the two sources can be symlinked side by side:
the four base-weight folders point at local disk, `loras` points at Drive.

In [ ]:
import os, shutil
COMFY_MODELS = '/content/ComfyUI/models'

links = {f'{COMFY_MODELS}/{s}': f'{HF_MODELS_DIR}/{s}' for s in HF_SUBDIRS}
links[f'{COMFY_MODELS}/loras'] = LORA_DIR          # ← Drive
links['/content/ComfyUI/output'] = OUTPUT_DIR      # ← Drive

for dst, src in links.items():
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst):
        if os.path.realpath(dst) == os.path.realpath(src):
            print(f'  ✅ {os.path.basename(dst)} → {src}')
            continue
        os.unlink(dst)
    elif os.path.isdir(dst):
        # Rescue anything ComfyUI already wrote here before replacing the dir.
        for entry in os.listdir(dst):
            target = os.path.join(src, entry)
            if not os.path.exists(target):
                shutil.move(os.path.join(dst, entry), target)
        shutil.rmtree(dst)
    elif os.path.exists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'  ✅ {os.path.basename(dst)} → {src}')
print('\n✅ Folders linked')

## Step 5 — Pull the base weights from Hugging Face

~34 GB straight to local disk. `hf_transfer` runs this multi-threaded at roughly
100–300 MB/s, so a cold session costs about **3–8 minutes**. Nothing here is written to
Drive. Re-running is free once the files are on disk.

In [ ]:
import os, shutil

# hf_transfer = multi-threaded downloads; it is what makes re-pulling 34 GB each
# session cheap enough to skip a Drive cache entirely.
try:
    import hf_transfer  # noqa: F401
except ImportError:
    !pip install -q hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

R22, R21 = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
STAGE = '/content/hf_stage'

MODELS = [
    ('Wan 2.2 I2V High Noise fp8 (~13GB)', R22,
     'split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors', 'diffusion_models'),
    ('Wan 2.2 I2V Low Noise fp8 (~13GB)', R22,
     'split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors', 'diffusion_models'),
    ('Text Encoder UMT5 fp8 (~6GB)', R21,
     'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
    ('VAE wan_2.1_vae (~242MB)', R21,
     'split_files/vae/wan_2.1_vae.safetensors', 'vae'),
    ('CLIP Vision (~1.2GB)', R21,
     'split_files/clip_vision/clip_vision_h.safetensors', 'clip_vision'),
]

for label, repo, remote, folder in MODELS:
    name = os.path.basename(remote)
    dest = f'{HF_MODELS_DIR}/{folder}/{name}'
    if os.path.exists(dest) and os.path.getsize(dest) > 1024:
        print(f'  ✅ Ready ({os.path.getsize(dest)/1024**3:.1f}GB): {label}')
        continue
    print(f'  ⬇️  Downloading: {label}')
    try:
        tmp = hf_hub_download(repo_id=repo, filename=remote, local_dir=STAGE)
    except Exception as e:
        print(f'     hf_hub_download failed ({e}); falling back to wget')
        tmp = f'{STAGE}/{name}'
        os.makedirs(STAGE, exist_ok=True)
        url = f'https://huggingface.co/{repo}/resolve/main/{remote}'
        !wget -q --show-progress -O "{tmp}" "{url}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(tmp, dest)   # instant: STAGE and HF_MODELS_DIR are both on local disk
    print(f'  ✅ Done ({os.path.getsize(dest)/1024**3:.1f}GB): {label}')

shutil.rmtree(STAGE, ignore_errors=True)
print('\n✅ Base weights ready at', HF_MODELS_DIR)

## Step 6 — *(Optional)* Lightning 4-step LoRAs — ~6× faster · **kept in Drive**

[lightx2v Wan2.2-Lightning](https://huggingface.co/lightx2v/Wan2.2-Lightning) LoRAs let you
render in **4–8 steps** at **CFG 1.0** instead of 20 steps. Separate LoRA per expert.

Unlike the base weights, these are downloaded **once into Drive** and reused every session —
so any LoRAs you add yourself (drop them in `MyDrive/ComfyUI_Wan/models/loras/`) show up here
too. Skip this cell for maximum quality; your own LoRAs are still linked either way.

In the workflow: add a `LoraLoaderModelOnly` on **each** expert (HIGH lora → high-noise model,
LOW lora → low-noise model), set steps **4–8**, CFG **1.0**, sampler **euler**, scheduler **simple**.

In [ ]:
import os, shutil
from huggingface_hub import HfApi, hf_hub_download

LORA_REPO = 'lightx2v/Wan2.2-Lightning'
try:
    files = HfApi().list_repo_files(LORA_REPO)
    cand = [f for f in files if f.endswith('.safetensors') and 'I2V' in f.upper()
            and '4STEP' in f.upper().replace('-', '').replace('_', '')]
    if not cand:
        cand = [f for f in files if f.endswith('.safetensors') and 'I2V' in f.upper()]
    picks = []
    for tag in ('HIGH', 'LOW'):
        m = [f for f in cand if tag in f.upper()]
        if m:
            picks.append(sorted(m)[0])
    picks = picks or cand[:2]
    print('Selected:', *picks, sep='\n  ')
    for src in picks:
        name = os.path.basename(src)
        dest = f'{LORA_DIR}/{name}'          # ← Drive: downloaded once, kept
        if os.path.exists(dest):
            print(f'  ✅ Already in Drive: {name}')
            continue
        shutil.copyfile(hf_hub_download(repo_id=LORA_REPO, filename=src), dest)
        print(f'  ✅ Downloaded to Drive: {name}')
except Exception as e:
    print('Could not auto-download Lightning LoRAs:', e)
    print('Install them via ComfyUI-Manager instead, or skip this step.')

have = sorted(f for f in os.listdir(LORA_DIR) if f.endswith('.safetensors'))
size = sum(os.path.getsize(f'{LORA_DIR}/{f}') for f in have) / 1024**3
print(f'\n✅ {len(have)} LoRA(s) in Drive ({size:.2f} GB) at {LORA_DIR}')
for f in have:
    print('   ·', f)

## Step 7 — Launch ComfyUI + public URL

Starts ComfyUI, waits until it's actually serving, then exposes it. Pick a `TUNNEL` method:

- **`colab`** *(default — no auth)*: opens ComfyUI as a **clickable "new window" link + an embedded iframe** inside this cell's output. ⚠️ Do **not** copy the raw `...prod.colab.dev` URL into a separate browser — it only works inside this Colab session and otherwise returns **HTTP 404**. Use the link/iframe this cell renders.
- **`cloudflare`**: quick `trycloudflare.com` tunnel that works in any browser. The cell first deletes any stale `~/.cloudflared` credentials (the usual cause of *"authentication"* errors) and installs a fresh binary.
- **`ngrok`**: paste a free authtoken from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

If ComfyUI itself fails to start, the tail of its log is shown. **Keep this cell running.**

In [ ]:
#@title Step 7 — Launch ComfyUI + tunnel { display-mode: "form" }
TUNNEL = "colab"  #@param ["colab", "cloudflare", "ngrok"]
NGROK_TOKEN = ""  #@param {type:"string"}

import os, re, time, subprocess, urllib.request

PORT = 8188
LOG = '/tmp/comfyui.log'

# --- start ComfyUI (kill any previous run first) ---
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# --enable-cors-header '*' relaxes cross-origin checks for proxied access.
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
     '--preview-method', 'auto', '--enable-cors-header', '*'],
    cwd='/content/ComfyUI', stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

print('Waiting for ComfyUI to start...')
ready = False
for _ in range(60):
    time.sleep(2)
    if comfy.poll() is not None:
        print('\n❌ ComfyUI exited. Last log lines:\n')
        print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
        raise SystemExit('ComfyUI failed to start — see log above.')
    try:
        if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2).status == 200:
            ready = True
            break
    except Exception:
        pass
if not ready:
    print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
    raise SystemExit('ComfyUI did not become ready in time — see log above.')
print('✅ ComfyUI is serving on :%d' % PORT)

def banner(url, note=''):
    print('\n' + '=' * 64)
    print('  🚀  Open ComfyUI:  ' + url)
    if note:
        print('  ' + note)
    print('=' * 64)

tunnel = None

if TUNNEL == 'colab':
    # Same-origin embed/link — avoids both the 404 (raw proxy URL pasted in a new
    # browser) and ComfyUI's 403 host/origin check. RECOMMENDED on Colab.
    from google.colab import output
    print('▶ Click this link to open ComfyUI in a new tab:')
    output.serve_kernel_port_as_window(PORT)
    print('\n▶ ...or use ComfyUI embedded right here:')
    output.serve_kernel_port_as_iframe(PORT, height='820')

elif TUNNEL == 'ngrok':
    # ngrok forwards Host == its own domain == Origin, so ComfyUI's host/origin
    # check passes. Reliable public URL that works in any browser.
    if not NGROK_TOKEN:
        raise SystemExit('Set NGROK_TOKEN in the form (free at dashboard.ngrok.com), or use TUNNEL="colab".')
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    banner(ngrok.connect(PORT, 'http').public_url, '(ngrok)')

elif TUNNEL == 'cloudflare':
    # Quick tunnel. ComfyUI v1.19+ may 403 ("non matching host and origin")
    # through a proxy; --http-host-header keeps the forwarded Host aligned, and a
    # stale cookie is the other common cause — open the link in an incognito tab.
    # If it still 403s, switch TUNNEL to "ngrok" or "colab".
    !rm -rf ~/.cloudflared
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate',
         '--url', f'http://127.0.0.1:{PORT}', '--http-host-header', f'127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_re = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    found = False
    t0 = time.time()
    for line in tunnel.stdout:
        m = url_re.search(line)
        if m:
            banner(m.group(0), '403? open in an incognito tab; still 403 → use TUNNEL="ngrok" or "colab"')
            found = True
            break
        if time.time() - t0 > 40:
            break
    if not found:
        print('⚠️  Cloudflare returned no URL. Set TUNNEL="colab" or "ngrok" and re-run.')

print('\n⏳ Keep this cell running. Interrupt to stop.')
try:
    comfy.wait()
except KeyboardInterrupt:
    comfy.terminate()
    if tunnel:
        tunnel.terminate()
    print('\n🛑 ComfyUI stopped')

## Step 8 — Run the I2V workflow

In the ComfyUI tab:
1. **Load template:** `Workflow → Browse Templates → Video → Wan2.2 14B I2V`.
2. Confirm the loaders point at your files: high/low noise `UNETLoader`s, `CLIPLoader` = `umt5_xxl_fp8_e4m3fn_scaled`, `VAELoader` = **`wan_2.1_vae.safetensors`**.
3. **Load Image** node = your first frame (use the *Copy Input Image* utility below, or upload via the UI).
4. Write the motion/scene in the positive prompt.
5. **Settings (A100):** 832×480 (fast) or 1280×720 (HQ); 81 frames @ 16 fps (~5 s); 20 steps, CFG 3.5. *With Lightning LoRAs:* 4–8 steps, CFG 1.0.
6. **Run.** Output MP4 lands in Drive at `ComfyUI_Wan/output/`.

**OOM on 40 GB?** Use 480p, fewer frames, and add `--lowvram` to the launch command in Step 7.

---
## 🔧 Utilities

### Copy an input image into ComfyUI

In [ ]:
import shutil, os
SOURCE = f'{INPUT_DIR}/my_image.jpg'  # update filename
if os.path.exists(SOURCE):
    shutil.copy(SOURCE, f'/content/ComfyUI/input/{os.path.basename(SOURCE)}')
    print(f'✅ Copied: {os.path.basename(SOURCE)} (pick it in the Load Image node)')
else:
    print(f'⚠️  Not found: {SOURCE}\n   Put the image in {INPUT_DIR}')

### Extract last frame (to chain clips)

In [ ]:
import shutil
CLIP_IN    = f'{OUTPUT_DIR}/wan22_i2v_00001.mp4'  # update filename
LAST_FRAME = f'{INPUT_DIR}/last_frame.jpg'
!ffmpeg -y -sseof -1 -i "{CLIP_IN}" -frames:v 1 "{LAST_FRAME}"
shutil.copy(LAST_FRAME, '/content/ComfyUI/input/last_frame.jpg')
print('✅ Last frame ready in ComfyUI input')

### Concatenate clips / add audio

In [ ]:
import glob
clips = sorted(glob.glob(f'{OUTPUT_DIR}/wan22_i2v_*.mp4'))
print(f'Found {len(clips)} clips')
if len(clips) >= 2:
    OUT = f'{OUTPUT_DIR}/wan22_i2v_combined.mp4'
    with open('/tmp/concat.txt', 'w') as f:
        for c in clips:
            f.write(f"file '{c}'\n")
    !ffmpeg -y -f concat -safe 0 -i /tmp/concat.txt -c copy "{OUT}"
    print(f'✅ Combined → {OUT}')
else:
    print('⚠️  Need ≥ 2 clips to concatenate')

### Reclaim Drive space (delete base weights from Drive — LoRAs are kept)

If you ran the original notebook, ~34 GB of base weights are still in
`MyDrive/ComfyUI_Wan/models`. This notebook no longer needs them. Lists them with sizes;
set `CONFIRM_DELETE = True` to remove them. **`loras/` is never touched**, and neither is
`output/`.

In [ ]:
#@title Reclaim Drive space { display-mode: "form" }
CONFIRM_DELETE = False  #@param {type:"boolean"}

import os
drive_models = f'{DRIVE_BASE}/models'
keep = os.path.realpath(LORA_DIR)
exts = ('.safetensors', '.ckpt', '.pt', '.bin')

found = []
for r, _, fs in os.walk(drive_models):
    if os.path.realpath(r) == keep or os.path.realpath(r).startswith(keep + os.sep):
        continue                                  # never touch LoRAs
    found += [(os.path.join(r, f), os.path.getsize(os.path.join(r, f)))
              for f in fs if f.endswith(exts)]

total = sum(s for _, s in found)
for p, s in sorted(found, key=lambda x: -x[1]):
    print(f'  {s/1024**3:7.2f} GB  {os.path.relpath(p, drive_models)}')
print(f'\n  {total/1024**3:7.2f} GB  TOTAL deletable (LoRAs excluded)')

if not found:
    print('\n✅ No base weights left in Drive.')
elif CONFIRM_DELETE:
    for p, _ in found:
        os.remove(p)
    print(f'\n✅ Deleted {len(found)} files — {total/1024**3:.1f} GB freed.')
    print('   Empty Drive Trash to actually reclaim the quota.')
else:
    print('\n(dry run — tick CONFIRM_DELETE to delete these files)')

---
## 📋 Prompt & troubleshooting tips

**Frames:** 81 ≈ 5s · 113 ≈ 7s · 145 ≈ 9s · 161 ≈ 10s (@16fps)

**Prompts** — Wan 2.2 follows direction well; be specific about motion and camera:
- Motion: `"she turns, walks forward, pauses and looks back"`
- Camera: `"slow push in"`, `"gentle pan right"`, `"tilt up to sky"`
- Quality: `"cinematic, smooth motion, film grain, 4k"`
- Negative: `"low quality, blurry, distorted, static, no motion, watermark, jitter"`

**Troubleshooting**
- **Cloudflare "authentication" / 403** → use `TUNNEL = "colab"` in Step 7 (no auth, no external service). The Cloudflare path also wipes stale `~/.cloudflared` creds, which are the usual cause.
- **VAE errors** → Load VAE must be `wan_2.1_vae.safetensors`, not the 2.2 VAE.
- **Missing nodes** → Step 3 does `git pull`; restart ComfyUI so core picks up Wan 2.2 nodes.
- **OOM** → 480p, fewer frames, `--lowvram` in Step 7.
- **Cell stops immediately** → Step 7 prints the ComfyUI log tail on failure; read it for the real error.

**Storage**
- **`No space left on device` in Step 5** → the base weights need ~34 GB on local disk plus download headroom. `Runtime → Disconnect and delete runtime` gives a clean disk.
- **Step 5 re-downloads every session** → expected. `/content` is wiped when the runtime recycles; that's the trade for keeping 34 GB out of Drive. Within one session it's a no-op.
- **A LoRA doesn't appear in ComfyUI** → it must be in `MyDrive/ComfyUI_Wan/models/loras/`; re-run Step 4, then `Refresh` in the ComfyUI menu.
- **`A Google Drive quota has been exceeded`** → only LoRAs and renders come from Drive here, so this is rare; if it happens, wait a few minutes rather than re-running Step 5.
- **Want the base weights durable too?** → use `wan22_i2v_comfyui_colab.ipynb`, which makes the whole model tree configurable (Drive, a GCS bucket, or any mounted path).